# Лабораторная работа 3. Коррекция опечаток

In [ ]:
import re
import random
import timeit
from collections import Counter
from difflib import get_close_matches
from zipfile import ZipFile

import pandas as pd
from IPython.display import display
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein, DamerauLevenshtein
from symspellpy import SymSpell, Verbosity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Три готовых метода исправления

In [ ]:
counts = Counter()
with ZipFile('../lab_2/data/news_science.zip') as archive:
    for name in sorted(archive.namelist()):
        if name.startswith('news_science/') and name.endswith('.txt'):
            text = archive.read(name).decode('utf-8')
            counts.update(re.findall(r'[а-яё]+', text.lower()))

vocabulary = dict(counts.most_common(10000))
words = list(vocabulary)
symspell = SymSpell(max_dictionary_edit_distance=2)
for word, frequency in vocabulary.items():
    symspell.create_dictionary_entry(word, frequency)

print('Слов в словаре:', len(words))

In [ ]:
def correct_difflib(word):
    matches = get_close_matches(word, words, n=1, cutoff=0.75)
    return matches[0] if matches else word


def correct_rapidfuzz(word):
    match = process.extractOne(word, words, scorer=Levenshtein.distance, score_cutoff=2)
    return match[0] if match else word


def correct_symspell(word):
    matches = symspell.lookup(word, Verbosity.TOP, max_edit_distance=2, include_unknown=True)
    return matches[0].term

In [ ]:
phrases = [
    'вот в инете откапал такую инеерсную статейку предлагаю вашему внимани',
    'может и в правду лутше тебе молчать чем пытаться сказать',
    'утром мы сидели как сычи а потом каааак начали работать'
]
methods = {
    'difflib': correct_difflib,
    'RapidFuzz': correct_rapidfuzz,
    'SymSpell': correct_symspell
}

for phrase in phrases:
    print('Исходная:', phrase)
    for name, correct in methods.items():
        print(name + ':', ' '.join(correct(word) for word in phrase.split()))
    print()
print('Словарь ограничен корпусом: исправления могут быть неверными.')

In [ ]:
sample_words = ' '.join(phrases).split()
times = []
for name, correct in methods.items():
    seconds = timeit.timeit(lambda: [correct(word) for word in sample_words], number=3)
    times.append({'Метод': name, 'Среднее время, мс': seconds / 3 * 1000})
pd.DataFrame(times)

## 2. Полное расстояние Дамерау — Левенштейна

In [ ]:
def damerau_levenshtein(a, b):
    n, m = len(a), len(b)
    limit = n + m
    d = [[0] * (m + 2) for _ in range(n + 2)]
    d[0][0] = limit
    for i in range(n + 1):
        d[i + 1][0] = limit
        d[i + 1][1] = i
    for j in range(m + 1):
        d[0][j + 1] = limit
        d[1][j + 1] = j

    last = {}  # Последняя строка, где встречалась буква.
    for i in range(1, n + 1):
        matched_column = 0
        for j in range(1, m + 1):
            previous_row = last.get(b[j - 1], 0)
            previous_column = matched_column
            cost = 1
            if a[i - 1] == b[j - 1]:
                cost = 0
                matched_column = j
            d[i + 1][j + 1] = min(
                d[i][j] + cost,
                d[i + 1][j] + 1,
                d[i][j + 1] + 1,
                d[previous_row][previous_column]
                + (i - previous_row - 1) + 1 + (j - previous_column - 1)
            )
        last[a[i - 1]] = i
    return d[n + 1][m + 1]

In [ ]:
pairs = [('инеерсную', 'интересную'), ('лутше', 'лучше'), ('внимани', 'вниманию'), ('кот', 'кто')]
comparison = []
for a, b in pairs:
    comparison.append({
        'Слово 1': a, 'Слово 2': b,
        'Своя функция': damerau_levenshtein(a, b),
        'RapidFuzz': DamerauLevenshtein.distance(a, b)
    })
display(pd.DataFrame(comparison))
repeats = 1000
seconds = timeit.timeit(lambda: [damerau_levenshtein(a, b) for a, b in pairs], number=repeats)
print('Среднее время для всех пар, мс:', seconds / repeats * 1000)

## 3. Опечатки по соседним клавишам

In [ ]:
rows = ['йцукенгшщзхъ', 'фывапролджэ', 'ячсмитьбю']
offsets = [0, 0.25, 0.75]
positions = {}
for row_number, row in enumerate(rows):
    for column, letter in enumerate(row):
        positions[letter] = (column + offsets[row_number], row_number)

neighbors = {}
for letter, (x, y) in positions.items():
    neighbors[letter] = []
    for other, (other_x, other_y) in positions.items():
        if other != letter and abs(y - other_y) <= 1 and abs(x - other_x) <= 1:
            neighbors[letter].append(other)


def make_typo(word):
    supported = []
    for i, letter in enumerate(word):
        if letter.lower() in neighbors:
            supported.append(i)
    if not supported:
        return word
    i = random.choice(supported)
    replacement = random.choice(neighbors[word[i].lower()])
    if word[i].isupper():
        replacement = replacement.upper()
    return word[:i] + replacement + word[i + 1:]

In [ ]:
random.seed(42)
for word in ['Клавиатура', 'студент', 'программа', '123']:
    print(word, '→', [make_typo(word) for _ in range(5)])

## 4. Сходство частотных векторов букв

In [ ]:
def compare_words(word1, word2):
    vectorizer = CountVectorizer(analyzer='char')
    vectors = vectorizer.fit_transform([word1, word2])
    return float(cosine_similarity(vectors[0], vectors[1])[0, 0])


vectorizer = CountVectorizer(analyzer='char')
vectors = vectorizer.fit_transform(['кот', 'ток'])
display(pd.DataFrame(vectors.toarray(), index=['кот', 'ток'],
                     columns=vectorizer.get_feature_names_out()))

for word1, word2 in [('кот', 'ток'), ('кот', 'кит'), ('кот', 'собака')]:
    print(word1, word2, round(compare_words(word1, word2), 3))
print('кот и ток имеют одинаковые векторы: порядок букв не учитывается.')